# LLM Experiment-3

In [16]:
!pip install transformers datasets scikit-learn -q

In [17]:
import pandas as pd
import numpy as np
import torch
import random

from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [18]:
from google.colab import files
uploaded = files.upload()

Saving creative_writing_dataset.csv to creative_writing_dataset.csv


In [19]:
df = pd.read_csv("creative_writing_dataset.csv")
df.head()

,id,text,label,label_id
0,1,The dragon circled above the silver towers of ...,Fantasy,0
1,2,Captain Reyes watched the twin suns set over M...,Science Fiction,1
2,3,"She traced his name in the fogged café window,...",Romance,2
3,4,"The knocking came again, slow and deliberate, ...",Horror,3
4,5,Detective Rowan noticed the clock had stopped ...,Mystery,4


In [20]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_id'],
        num_rows: 320
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_id'],
        num_rows: 80
    })
})

In [21]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [22]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

In [23]:
tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["id", "text", "label"])
tokenized_dataset.set_format("torch")
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 320
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 80
    })
})

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [33]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [34]:
pretrained_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

trainer_before = Trainer(
    model=pretrained_model,
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

metrics_before = trainer_before.evaluate()
metrics_before

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 1.6049964427947998,
 'eval_model_preparation_time': 0.0107,
 'eval_accuracy': 0.2375,
 'eval_precision': 0.09669117647058824,
 'eval_recall': 0.2375,
 'eval_f1': 0.12410554561717353,
 'eval_runtime': 0.8011,
 'eval_samples_per_second': 99.86,
 'eval_steps_per_second': 12.482}

In [35]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

In [37]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,1.087233,0.787500,0.841989,0.787500,0.759119
2,No log,0.441553,0.937500,0.943342,0.937500,0.936305
3,No log,0.217045,0.937500,0.946569,0.937500,0.937405
4,No log,0.190954,0.950000,0.950827,0.950000,0.950069
5,No log,0.203755,0.950000,0.950827,0.950000,0.950069
6,No log,0.234898,0.950000,0.950827,0.950000,0.950069
7,No log,0.219633,0.962500,0.962500,0.962500,0.962500
8,No log,0.228776,0.950000,0.950827,0.950000,0.950069
9,No log,0.233912,0.950000,0.950827,0.950000,0.950069
10,No log,0.234001,0.950000,0.950827,0.950000,0.950069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=400, training_loss=0.26275842666625976, metrics={'train_runtime': 225.4071, 'train_samples_per_second': 14.197, 'train_steps_per_second': 1.775, 'total_flos': 210494513971200.0, 'train_loss': 0.26275842666625976, 'epoch': 10.0})

In [38]:
metrics_after = trainer.evaluate()
metrics_after

{'eval_loss': 0.19126972556114197,
 'eval_accuracy': 0.95,
 'eval_precision': 0.9508272058823529,
 'eval_recall': 0.95,
 'eval_f1': 0.9500691244239631,
 'eval_runtime': 0.7018,
 'eval_samples_per_second': 113.994,
 'eval_steps_per_second': 14.249,
 'epoch': 10.0}

In [41]:
keys = ["accuracy", "precision", "recall", "f1"]

table = pd.DataFrame({
    "Metric": keys,
    "Before Fine-Tuning": [metrics_before["eval_" + k] for k in keys],
    "After Fine-Tuning": [metrics_after["eval_" + k] for k in keys]
})

print(table)

      Metric  Before Fine-Tuning  After Fine-Tuning
0   accuracy            0.237500           0.950000
1  precision            0.096691           0.950827
2     recall            0.237500           0.950000
3         f1            0.124106           0.950069


In [42]:
trainer.save_model("fine_tuned_bert_creative_writing")
tokenizer.save_pretrained("fine_tuned_bert_creative_writing")

# Optional: Zip for download
# !zip -r fine_tuned_bert_creative_writing.zip fine_tuned_bert_creative_writing

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fine_tuned_bert_creative_writing/tokenizer_config.json',
 'fine_tuned_bert_creative_writing/tokenizer.json')

In [43]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

model_path = "fine_tuned_bert_creative_writing"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

id_to_label = {
    0: "Fantasy",
    1: "Science Fiction",
    2: "Romance",
    3: "Horror",
    4: "Mystery"
}

def predict_genre(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    return predictions.item()


print("Welcome to the Creative Writing Genre Predictor CLI!")
print("Enter your text, or type 'quit' to exit.")

while True:
    user_input = input("\nEnter text: ")
    if user_input.lower() == 'quit':
        break

    if not user_input.strip(): # Handle empty input
        print("Please enter some text.")
        continue

    predicted_id = predict_genre(user_input)
    predicted_label = id_to_label.get(predicted_id, "Unknown")
    print(f"Predicted Genre: {predicted_label}")

print("Exiting CLI. Goodbye!")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Welcome to the Creative Writing Genre Predictor CLI!
Enter your text, or type 'quit' to exit.

Enter text: I saw a dragon on the top of the Astronomy Tower in Hogwarts.
Predicted Genre: Fantasy

Enter text: He watched her laugh across the crowded room, feeling an ache he hadn’t known existed.
Predicted Genre: Romance

Enter text: On the surface of Kepler-442b, the sky burned with violet storms.
Predicted Genre: Science Fiction

Enter text: Time is not a river, but an ocean
Predicted Genre: Fantasy

Enter text: The envelope had no return address, and the handwriting was unfamiliar. Inside, the note simply read: 'Meet me where the clocks never tick.'
Predicted Genre: Mystery

Enter text: The ominous wind ran a shiver down his spine
Predicted Genre: Horror

Enter text: quit
Exiting CLI. Goodbye!
